# 基于动态图神经网络（DGCNN）的 SEED 脑电情绪识别与域适应

本 Notebook 在 **Google Colab** 中直接读取已上传至 Google Drive 的上海交大 **SEED（62 导三分类）** 完整数据集，执行：
1. 五频段 DE + PSD 节点特征与 3D 电极距离初始化邻接矩阵构建；
2. **被试内（Within-Subject 9/6 划分）** 对照实验：SVM、MLP、TSception、DGCNN；
3. **跨被试（LOSO 15 折）** 无监督域适应消融实验：DGCNN（未对齐）、DGCNN + MMD、DGCNN + DANN、DGCNN + MMD + DANN；
4. 学习邻接矩阵的脑区连接权重统计（验证额叶与颞叶的主导贡献），并将所有结果持久化至云端 `results/` 目录。

- **推荐运行时**：`代码执行程序 -> 更改运行时类型 -> T4 GPU`
- **云端项目路径**：`/content/drive/MyDrive/BCI_Portfolio/03_SEED_Emotion_DGCNN_DA`

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/BCI_Portfolio/03_SEED_Emotion_DGCNN_DA')
assert PROJECT_DIR.exists(), f'未找到项目目录: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

seed_dir = PROJECT_DIR / 'data' / 'SEED' / 'ExtractedFeatures'
mat_files = list(seed_dir.glob('*.mat')) if seed_dir.exists() else []
print('工作目录:', os.getcwd())
print(f'已检测到云端 SEED 数据文件: {len(mat_files)} 个 .mat 文件')

In [ ]:
!pip install -q -r requirements.txt
import torch
print(f'PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')

In [ ]:
from src.utils import load_config, set_seed, setup_logger
from src.evaluation import run_seed_benchmark

RUN_MODE = 'full'  # 'full' = 全部 15 名被试（被试内 + 15 折 LOSO）；'quick' = 前 5 名被试快速验证

cfg = load_config(PROJECT_DIR / 'configs' / 'config.yaml')
cfg['project']['data_dir'] = str(PROJECT_DIR / cfg['project']['data_dir'])
cfg['project']['results_dir'] = str(PROJECT_DIR / cfg['project']['results_dir'])

set_seed(int(cfg['project']['seed']))
setup_logger('seed_dgcnn', log_file=PROJECT_DIR / 'results' / 'run.log')

targets = cfg['dataset']['subjects'] if RUN_MODE == 'full' else cfg['dataset']['quick_subjects']
summary = run_seed_benchmark(
    target_subjects=targets,
    cfg=cfg,
    run_within=True,
    run_loso=True,
)

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

results_dir = PROJECT_DIR / 'results'
print('云端已保存结果文件:', sorted(os.listdir(results_dir)))
md_path = results_dir / 'benchmark_report.md'
if md_path.exists():
    display(Markdown(md_path.read_text(encoding='utf-8')))